In [2]:
import os
import sys
import pandas as pd
from typing import Tuple

In [3]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [4]:
from utility.data_log_functions import DataLogHelper

In [5]:
def compare_multiple_code_generation_logs(res_dir: str, filter: Tuple[str] = (), anti_filter: Tuple[str] = ()):
    
    if filter is None:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv"))]
    else:
        csv_logs = [f for f in os.listdir(res_dir) if (
            os.path.isfile(os.path.join(res_dir, f)) and 
            f.endswith(".csv") and 
            all(sub in f for sub in filter)) and
            all(sub not in f for sub in anti_filter)
            ]

    count = 0

    log_file_names = [csv_file_name.replace('.csv', '') for csv_file_name in csv_logs]

    results_df = pd.DataFrame(columns=log_file_names, index = log_file_names)
    for file_name in log_file_names:
        results_df.loc[file_name, file_name] = float('nan')

    while count < (len(csv_logs)**2-len(csv_logs)//2):
        log1_file_name = csv_logs.pop()
        for log2_file_name in csv_logs:
            log1_file_path = os.path.join(res_dir, log1_file_name)
            log2_file_path = os.path.join(res_dir, log2_file_name)

            print(log1_file_path)
            print(log2_file_path)


            log1 = pd.read_csv(log1_file_path)
            log2 = pd.read_csv(log2_file_path) 
            print(log1_file_name, log2_file_name)
            log1_inconsistencies, log2_inconsistencies = DataLogHelper.compare_code_generation_dataframe_results(log1=log1, log2=log2)

            results_df.loc[log1_file_name.replace('.csv', ''), log2_file_name.replace('.csv', '')] = log1_inconsistencies
            results_df.loc[log2_file_name.replace('.csv', ''), log1_file_name.replace('.csv', '')] = log2_inconsistencies

        count += 1
    
    return results_df

In [7]:
res_dir = proj_dir + "/results/code_inconsistencies/output_prediction/mistral"
res_dir = proj_dir + "/results/mcq_inconsistency/code_completion/mistral"

res = compare_multiple_code_generation_logs(res_dir=res_dir, filter=( ), anti_filter = ())
res_df = pd.DataFrame(res)

print(res_df)

/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/results/mcq_inconsistency/code_completion/mistral/CodeMMLU_MCQ_code_completion_mistral-small-2506_zero_shot_None_sequential mutation.csv
/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/results/mcq_inconsistency/code_completion/mistral/CodeMMLU_MCQ_code_completion_mistral-small-2506_zero_shot_None_random mutation.csv
CodeMMLU_MCQ_code_completion_mistral-small-2506_zero_shot_None_sequential mutation.csv CodeMMLU_MCQ_code_completion_mistral-small-2506_zero_shot_None_random mutation.csv
Starting comparison of 164 tasks...
Task CodeMMLU32: log1 succeeded, log2 failed (AssertionError > )
Task CodeMMLU35: log1 succeeded, log2 failed (AssertionError > )
Task CodeMMLU58: log1 succeeded, log2 failed (AssertionError > )
Task CodeMMLU72: log1 succeeded, log2 failed (AssertionError > )
Task CodeMMLU145: log1 succeeded, log2 failed (AssertionError > )
Task CodeMMLU157: log1 succeeded, log2 failed (AssertionError > )
